# MNIST Neural Network: Intermediate Level 2
## Component-Based Object-Oriented Design

**Complexity Level**: Intermediate  
**Architectural Style**: Multiple cohesive classes with clear responsibilities  
**Focus**: Strategy Pattern for activations, clean separation of concerns

---

## The Problem: Handwritten Digit Recognition

The MNIST dataset presents a classic machine learning challenge: given an image of a handwritten digit, determine which digit (0-9) it represents.

**Inputs**: Each image is a 28×28 pixel grayscale image, which we flatten into a single list of 784 values. Each pixel value ranges from 0 (black) to 255 (white), though we typically normalise these to the range 0.0 to 1.0 by dividing by 255. So our neural network receives 784 floating-point numbers as input for each image.

**Outputs**: The network produces 10 values, one for each possible digit (0 through 9). These outputs represent the network's confidence that the input image shows each respective digit. We typically interpret these as probabilities (summing to 1.0 after applying softmax), and the digit with the highest probability becomes our prediction. For training, we compare these outputs against "one-hot" encoded labels where the correct digit has value 1.0 and all others have 0.0.

**The Task**: Build a function that maps 784 input values to 10 output values, learning the patterns that distinguish each digit through repeated exposure to labelled examples.

---

## Why This Approach?

This notebook uses the **Strategy Pattern** for activation functions and cleanly separates:

- **Activation functions**: As pluggable objects (Strategy Pattern)
- **Dense Layer**: Handles weights, computation
- **Network**: Coordinates layers
- **Loss functions**: As separate objects
- **Optimiser**: Handles parameter updates

**Design Philosophy**: "Each class does one thing well."

**Trade-offs**:
- More classes = more indirection to understand
- But: easier to test, extend, and modify each component
- Still simpler than full Node/Connection decomposition

**Alternative**: Could add more patterns (Factory for layers, Observer for logging).

In [ ]:
from abc import ABC, abstractmethod
from typing import List, Tuple, Optional
import math
import random

random.seed(42)

---

## Part 1: Activation Functions (Strategy Pattern)

The **Strategy Pattern** allows us to:
- Define a family of algorithms (activation functions)
- Encapsulate each one in a class
- Make them interchangeable

**Benefits**:
- Easy to add new activations without changing Layer code
- Each activation is self-documenting (has a name, can describe itself)
- Can be tested independently

**Alternative**: Simple functions (see previous notebook). Simpler but less extensible.

In [ ]:
class Activation(ABC):
    """
    Abstract base class for activation functions.
    
    Strategy Pattern: Defines interface that all activation functions must implement.
    
    Any new activation function must provide:
    - forward(): Apply the activation
    - derivative(): Compute gradient for backpropagation
    - name: Human-readable identifier
    """
    
    @property
    @abstractmethod
    def name(self) -> str:
        """Return name of activation function."""
        pass
    
    @abstractmethod
    def forward(self, x: float) -> float:
        """
        Apply activation function to input.
        
        Args:
            x: Pre-activation value
        
        Returns:
            Activated value
        """
        pass
    
    @abstractmethod
    def derivative(self, x: float) -> float:
        """
        Compute derivative for backpropagation.
        
        Args:
            x: Pre-activation value (same as input to forward)
        
        Returns:
            Derivative value
        """
        pass
    
    def forward_batch(self, values: List[float]) -> List[float]:
        """Apply activation to list of values."""
        return [self.forward(x) for x in values]
    
    def derivative_batch(self, values: List[float]) -> List[float]:
        """Compute derivative for list of values."""
        return [self.derivative(x) for x in values]
    
    def __repr__(self):
        return f"{self.name}()"

In [ ]:
class ReLU(Activation):
    """
    Rectified Linear Unit: f(x) = max(0, x)
    
    Pros:
    - Fast to compute
    - Helps avoid vanishing gradients
    - Sparse activation (many zeros)
    
    Cons:
    - 'Dying ReLU' problem (neurons can get stuck at 0)
    - Not differentiable at x=0 (but we define it as 0)
    """
    
    @property
    def name(self) -> str:
        return "ReLU"
    
    def forward(self, x: float) -> float:
        return max(0.0, x)
    
    def derivative(self, x: float) -> float:
        return 1.0 if x > 0 else 0.0


class LeakyReLU(Activation):
    """
    Leaky ReLU: f(x) = x if x > 0, else alpha * x
    
    Addresses dying ReLU by allowing small gradients for negative inputs.
    
    Common alpha values: 0.01 (default), 0.1, 0.2
    """
    
    def __init__(self, alpha: float = 0.01):
        self.alpha = alpha
    
    @property
    def name(self) -> str:
        return f"LeakyReLU(alpha={self.alpha})"
    
    def forward(self, x: float) -> float:
        return x if x > 0 else self.alpha * x
    
    def derivative(self, x: float) -> float:
        return 1.0 if x > 0 else self.alpha


class Sigmoid(Activation):
    """
    Sigmoid: f(x) = 1 / (1 + e^(-x))
    
    Pros:
    - Outputs bounded in (0, 1)
    - Smooth, differentiable everywhere
    
    Cons:
    - Vanishing gradients for large |x|
    - Outputs not zero-centred
    - Slower than ReLU
    """
    
    @property
    def name(self) -> str:
        return "Sigmoid"
    
    def forward(self, x: float) -> float:
        x = max(-500, min(500, x))  # Prevent overflow
        return 1.0 / (1.0 + math.exp(-x))
    
    def derivative(self, x: float) -> float:
        s = self.forward(x)
        return s * (1.0 - s)


class Tanh(Activation):
    """
    Hyperbolic tangent: f(x) = (e^x - e^(-x)) / (e^x + e^(-x))
    
    Pros:
    - Zero-centred (outputs in (-1, 1))
    - Stronger gradients than sigmoid
    
    Cons:
    - Still has vanishing gradient problem
    """
    
    @property
    def name(self) -> str:
        return "Tanh"
    
    def forward(self, x: float) -> float:
        x = max(-500, min(500, x))
        return math.tanh(x)
    
    def derivative(self, x: float) -> float:
        t = self.forward(x)
        return 1.0 - t * t

In [ ]:
class Softmax:
    """
    Softmax activation for output layer.
    
    Converts logits to probability distribution.
    NOT a subclass of Activation because it operates on entire vector.
    
    Design choice: Separate class, not subclass of Activation.
    Alternative: Could make Activation handle both cases.
    """
    
    @property
    def name(self) -> str:
        return "Softmax"
    
    def forward(self, values: List[float]) -> List[float]:
        """Convert logits to probabilities."""
        max_val = max(values)
        exp_values = [math.exp(v - max_val) for v in values]
        total = sum(exp_values)
        return [ev / total for ev in exp_values]
    
    def __repr__(self):
        return "Softmax()"

In [ ]:
# Test activation functions
print("Activation function comparison:")
print("=" * 50)

activations = [ReLU(), LeakyReLU(0.1), Sigmoid(), Tanh()]
test_values = [-2.0, -0.5, 0.0, 0.5, 2.0]

for act in activations:
    print(f"\n{act.name}:")
    for x in test_values:
        y = act.forward(x)
        dy = act.derivative(x)
        print(f"  f({x:5.1f}) = {y:7.4f}, f'({x:5.1f}) = {dy:6.4f}")

---

## Part 2: Loss Functions

Loss functions measure prediction error. Separating them allows easy experimentation.

**Design Choice**: Loss class with static methods.

**Alternative**: Separate classes per loss (overkill for simple cases).

In [ ]:
class Loss:
    """
    Collection of loss functions.
    
    Each loss function has:
    - forward: Compute the loss value
    - gradient: Compute gradient for backpropagation
    
    Design choice: Static methods in single class.
    Alternative: Abstract Loss base with CrossEntropy, MSE subclasses.
    """
    
    @staticmethod
    def cross_entropy(predicted: List[float], target: List[float]) -> float:
        """
        Cross-entropy loss for classification.
        
        Formula: -sum(target[i] * log(predicted[i]))
        
        Works with softmax output and one-hot targets.
        """
        epsilon = 1e-15
        return -sum(
            t * math.log(max(p, epsilon))
            for p, t in zip(predicted, target)
            if t > 0
        )
    
    @staticmethod
    def cross_entropy_gradient(predicted: List[float], target: List[float]) -> List[float]:
        """
        Gradient of cross-entropy loss with respect to softmax input.
        
        The beautiful property of softmax + cross-entropy:
        gradient = predicted - target
        """
        return [p - t for p, t in zip(predicted, target)]
    
    @staticmethod
    def mse(predicted: List[float], target: List[float]) -> float:
        """
        Mean Squared Error loss.
        
        Formula: mean((predicted[i] - target[i])^2)
        
        Alternative to cross-entropy, works for regression.
        """
        n = len(predicted)
        return sum((p - t) ** 2 for p, t in zip(predicted, target)) / n
    
    @staticmethod
    def mse_gradient(predicted: List[float], target: List[float]) -> List[float]:
        """
        Gradient of MSE loss.
        
        Formula: 2 * (predicted - target) / n
        """
        n = len(predicted)
        return [2 * (p - t) / n for p, t in zip(predicted, target)]

---

## Part 3: Weight Initialiser

Proper initialisation is crucial for training. We encapsulate initialisation strategies.

**Design Choice**: Static methods grouped in Initialiser class.

**Alternative**: Factory functions (simpler), or Strategy Pattern (more extensible).

In [ ]:
class Initialiser:
    """
    Weight initialisation strategies.
    
    Good initialisation helps:
    - Prevent vanishing/exploding gradients
    - Enable faster convergence
    - Break symmetry between neurons
    """
    
    @staticmethod
    def xavier(fan_in: int, fan_out: int) -> List[List[float]]:
        """
        Xavier/Glorot initialisation.
        
        Best for: Sigmoid, Tanh activations
        
        Formula: uniform(-limit, limit) where limit = sqrt(6 / (fan_in + fan_out))
        
        Maintains variance of activations across layers.
        """
        limit = math.sqrt(6.0 / (fan_in + fan_out))
        return [
            [random.uniform(-limit, limit) for _ in range(fan_out)]
            for _ in range(fan_in)
        ]
    
    @staticmethod
    def he(fan_in: int, fan_out: int) -> List[List[float]]:
        """
        He/Kaiming initialisation.
        
        Best for: ReLU and variants
        
        Formula: normal(0, sqrt(2/fan_in)) or uniform with limit = sqrt(6/fan_in)
        
        Accounts for ReLU zeroing negative values.
        """
        limit = math.sqrt(6.0 / fan_in)
        return [
            [random.uniform(-limit, limit) for _ in range(fan_out)]
            for _ in range(fan_in)
        ]
    
    @staticmethod
    def uniform(fan_in: int, fan_out: int, low: float = -0.5, high: float = 0.5) -> List[List[float]]:
        """
        Simple uniform random initialisation.
        
        Basic but often sufficient for small networks.
        """
        return [
            [random.uniform(low, high) for _ in range(fan_out)]
            for _ in range(fan_in)
        ]
    
    @staticmethod
    def zeros(fan_in: int, fan_out: int) -> List[List[float]]:
        """
        All zeros (generally not recommended).
        
        Problem: All neurons compute the same thing (symmetry).
        Only useful for biases.
        """
        return [[0.0] * fan_out for _ in range(fan_in)]

---

## Part 4: Dense Layer

The `DenseLayer` class represents a fully-connected layer.

**Key Responsibilities**:
- Store and manage weights/biases
- Compute forward pass
- Compute gradients during backward pass

**Does NOT handle**: Parameter updates (that's the Optimiser's job)

In [ ]:
class DenseLayer:
    """
    A fully-connected (dense) neural network layer.
    
    Computes: output = activation(W*input + bias)
    
    Responsibilities:
    - Store weights and biases
    - Forward pass computation
    - Gradient computation (not application - that's Optimiser's job)
    
    Design choice: Activation is injected (Strategy Pattern).
    Alternative: String name looked up internally.
    """
    
    def __init__(self,
                 input_size: int,
                 output_size: int,
                 activation: Activation,
                 init_method: str = 'he'):
        """
        Create dense layer.
        
        Args:
            input_size: Number of inputs
            output_size: Number of outputs (neurons)
            activation: Activation function object
            init_method: 'he', 'xavier', or 'uniform'
        """
        self.input_size = input_size
        self.output_size = output_size
        self.activation = activation
        
        # Initialise weights
        if init_method == 'he':
            self.weights = Initialiser.he(input_size, output_size)
        elif init_method == 'xavier':
            self.weights = Initialiser.xavier(input_size, output_size)
        else:
            self.weights = Initialiser.uniform(input_size, output_size)
        
        self.biases = [0.0] * output_size
        
        # Cache for backpropagation
        self.input_cache = None
        self.z_cache = None  # Pre-activation values
        
        # Gradient storage
        self.weight_grads = None
        self.bias_grads = None
    
    def forward(self, inputs: List[float]) -> List[float]:
        """
        Compute layer output.
        
        Steps:
        1. Compute weighted sum: z = W*x + b
        2. Apply activation: output = activation(z)
        3. Cache values for backpropagation
        
        Args:
            inputs: Input vector
        
        Returns:
            Output vector after activation
        """
        self.input_cache = inputs
        
        # Compute z = Wx + b
        z = []
        for j in range(self.output_size):
            total = self.biases[j]
            for i in range(self.input_size):
                total += inputs[i] * self.weights[i][j]
            z.append(total)
        
        self.z_cache = z
        
        # Apply activation
        return self.activation.forward_batch(z)
    
    def backward(self, upstream_gradient: List[float]) -> List[float]:
        """
        Compute gradients and return gradient for previous layer.
        
        Args:
            upstream_gradient: Gradient from next layer
        
        Returns:
            Gradient to pass to previous layer
        
        Side effects:
            Stores weight_grads and bias_grads
        """
        # Gradient through activation
        activation_grads = self.activation.derivative_batch(self.z_cache)
        delta = [
            upstream_gradient[j] * activation_grads[j]
            for j in range(self.output_size)
        ]
        
        # Compute weight gradients: dW = input^T * delta
        self.weight_grads = [
            [self.input_cache[i] * delta[j] for j in range(self.output_size)]
            for i in range(self.input_size)
        ]
        
        # Bias gradients = delta
        self.bias_grads = delta[:]
        
        # Gradient for previous layer: W^T * delta
        downstream_gradient = []
        for i in range(self.input_size):
            grad = sum(
                self.weights[i][j] * delta[j]
                for j in range(self.output_size)
            )
            downstream_gradient.append(grad)
        
        return downstream_gradient
    
    def get_parameters(self) -> Tuple[List[List[float]], List[float]]:
        """Return weights and biases."""
        return self.weights, self.biases
    
    def get_gradients(self) -> Tuple[List[List[float]], List[float]]:
        """Return computed gradients."""
        return self.weight_grads, self.bias_grads
    
    def num_parameters(self) -> int:
        """Count trainable parameters."""
        return self.input_size * self.output_size + self.output_size
    
    def __repr__(self):
        return f"DenseLayer({self.input_size} -> {self.output_size}, {self.activation.name})"

In [ ]:
class OutputLayer(DenseLayer):
    """
    Output layer with softmax activation.
    
    Specialised for classification tasks.
    Uses softmax and has simplified backward pass.
    """
    
    def __init__(self, input_size: int, num_classes: int, init_method: str = 'xavier'):
        # Manually initialise instead of calling super().__init__
        # because we use Softmax (not Activation subclass)
        self.input_size = input_size
        self.output_size = num_classes
        self.softmax = Softmax()
        self.activation = self.softmax  # For compatibility
        
        if init_method == 'xavier':
            self.weights = Initialiser.xavier(input_size, num_classes)
        else:
            self.weights = Initialiser.he(input_size, num_classes)
        
        self.biases = [0.0] * num_classes
        self.input_cache = None
        self.z_cache = None
        self.output_cache = None
        self.weight_grads = None
        self.bias_grads = None
    
    def forward(self, inputs: List[float]) -> List[float]:
        """Compute softmax output."""
        self.input_cache = inputs
        
        # Compute z = Wx + b
        z = []
        for j in range(self.output_size):
            total = self.biases[j]
            for i in range(self.input_size):
                total += inputs[i] * self.weights[i][j]
            z.append(total)
        
        self.z_cache = z
        self.output_cache = self.softmax.forward(z)
        return self.output_cache
    
    def backward_with_target(self, target: List[float]) -> List[float]:
        """
        Backward pass using target directly.
        
        For softmax + cross-entropy, gradient = output - target.
        """
        delta = [self.output_cache[j] - target[j] for j in range(self.output_size)]
        
        self.weight_grads = [
            [self.input_cache[i] * delta[j] for j in range(self.output_size)]
            for i in range(self.input_size)
        ]
        self.bias_grads = delta[:]
        
        # Gradient for previous layer
        downstream_gradient = []
        for i in range(self.input_size):
            grad = sum(self.weights[i][j] * delta[j] for j in range(self.output_size))
            downstream_gradient.append(grad)
        
        return downstream_gradient
    
    def predict(self) -> int:
        """Return predicted class."""
        return self.output_cache.index(max(self.output_cache))
    
    def __repr__(self):
        return f"OutputLayer({self.input_size} -> {self.output_size}, Softmax)"

---

## Part 5: Optimiser

Optimisers handle parameter updates. Separating this allows:
- Easy switching between SGD, Momentum, Adam, etc.
- Network doesn't need to know update rules

**Design Choice**: Optimiser holds reference to layers and updates them.

**Alternative**: Return new parameters (more functional style).

In [ ]:
class SGD:
    """
    Stochastic Gradient Descent optimiser.
    
    Update rule: parameter -= learning_rate * gradient
    
    This is the simplest optimiser. More advanced options:
    - SGD with momentum
    - RMSprop
    - Adam
    """
    
    def __init__(self, layers: List[DenseLayer], learning_rate: float = 0.01):
        """
        Initialise optimiser.
        
        Args:
            layers: List of layers to optimise
            learning_rate: Step size for updates
        """
        self.layers = layers
        self.learning_rate = learning_rate
    
    def step(self):
        """
        Apply gradient descent update to all layers.
        
        Assumes backward() has been called on all layers.
        """
        for layer in self.layers:
            weight_grads, bias_grads = layer.get_gradients()
            
            # Update weights
            for i in range(layer.input_size):
                for j in range(layer.output_size):
                    layer.weights[i][j] -= self.learning_rate * weight_grads[i][j]
            
            # Update biases
            for j in range(layer.output_size):
                layer.biases[j] -= self.learning_rate * bias_grads[j]

In [ ]:
class SGDMomentum:
    """
    SGD with momentum.
    
    Update rule:
    velocity = momentum * velocity - learning_rate * gradient
    parameter += velocity
    
    Momentum helps:
    - Accelerate convergence
    - Smooth out noisy gradients
    - Escape local minima
    
    Alternative optimiser demonstrating the design's extensibility.
    """
    
    def __init__(self, 
                 layers: List[DenseLayer], 
                 learning_rate: float = 0.01,
                 momentum: float = 0.9):
        self.layers = layers
        self.learning_rate = learning_rate
        self.momentum = momentum
        
        # Initialise velocity terms
        self.weight_velocities = []
        self.bias_velocities = []
        
        for layer in layers:
            w_vel = [[0.0] * layer.output_size for _ in range(layer.input_size)]
            b_vel = [0.0] * layer.output_size
            self.weight_velocities.append(w_vel)
            self.bias_velocities.append(b_vel)
    
    def step(self):
        """Apply momentum update."""
        for idx, layer in enumerate(self.layers):
            weight_grads, bias_grads = layer.get_gradients()
            
            # Update weights with momentum
            for i in range(layer.input_size):
                for j in range(layer.output_size):
                    self.weight_velocities[idx][i][j] = (
                        self.momentum * self.weight_velocities[idx][i][j]
                        - self.learning_rate * weight_grads[i][j]
                    )
                    layer.weights[i][j] += self.weight_velocities[idx][i][j]
            
            # Update biases with momentum
            for j in range(layer.output_size):
                self.bias_velocities[idx][j] = (
                    self.momentum * self.bias_velocities[idx][j]
                    - self.learning_rate * bias_grads[j]
                )
                layer.biases[j] += self.bias_velocities[idx][j]

---

## Part 6: Neural Network

The `NeuralNetwork` class orchestrates all components.

In [ ]:
class NeuralNetwork:
    """
    Complete neural network coordinating layers and training.
    
    This class is intentionally thin - it coordinates components
    rather than implementing logic itself.
    
    Design philosophy: Composition over inheritance.
    """
    
    def __init__(self, layers: List[DenseLayer]):
        """
        Create network from layers.
        
        Args:
            layers: List of layers (last should be OutputLayer)
        """
        self.layers = layers
        self._validate()
    
    def _validate(self):
        """Validate layer compatibility."""
        for i in range(len(self.layers) - 1):
            if self.layers[i].output_size != self.layers[i+1].input_size:
                raise ValueError(f"Layer dimension mismatch at {i}")
        
        if not isinstance(self.layers[-1], OutputLayer):
            raise TypeError("Last layer must be OutputLayer")
    
    def forward(self, inputs: List[float]) -> List[float]:
        """Forward pass through network."""
        current = inputs
        for layer in self.layers:
            current = layer.forward(current)
        return current
    
    def backward(self, target: List[float]) -> None:
        """
        Backward pass - compute all gradients.
        
        Does NOT update parameters (that's optimiser's job).
        """
        # Output layer uses special backward
        gradient = self.layers[-1].backward_with_target(target)
        
        # Hidden layers
        for i in range(len(self.layers) - 2, -1, -1):
            gradient = self.layers[i].backward(gradient)
    
    def predict(self, inputs: List[float]) -> int:
        """Predict class for input."""
        self.forward(inputs)
        return self.layers[-1].predict()
    
    def predict_proba(self, inputs: List[float]) -> List[float]:
        """Get prediction probabilities."""
        return self.forward(inputs)
    
    def summary(self):
        """Print network summary."""
        print("Neural Network")
        print("=" * 50)
        total = 0
        for i, layer in enumerate(self.layers):
            params = layer.num_parameters()
            total += params
            print(f"Layer {i}: {layer} ({params:,} params)")
        print("=" * 50)
        print(f"Total: {total:,} parameters")

---

## Part 7: Trainer

Manages the training loop, using network and optimiser.

In [ ]:
class Trainer:
    """
    Handles training loop.
    
    Coordinates network and optimiser.
    """
    
    def __init__(self, network: NeuralNetwork, optimiser):
        self.network = network
        self.optimiser = optimiser
    
    @staticmethod
    def _one_hot(label: int, num_classes: int) -> List[float]:
        vec = [0.0] * num_classes
        vec[label] = 1.0
        return vec
    
    def train_step(self, inputs: List[float], label: int) -> Tuple[float, bool]:
        """Train on single example."""
        num_classes = self.network.layers[-1].output_size
        target = self._one_hot(label, num_classes)
        
        # Forward
        output = self.network.forward(inputs)
        loss = Loss.cross_entropy(output, target)
        predicted = output.index(max(output))
        
        # Backward
        self.network.backward(target)
        
        # Update
        self.optimiser.step()
        
        return loss, (predicted == label)
    
    def train_epoch(self, data, labels):
        total_loss = 0.0
        correct = 0
        
        for inputs, label in zip(data, labels):
            loss, is_correct = self.train_step(inputs, label)
            total_loss += loss
            correct += int(is_correct)
        
        return total_loss / len(data), correct / len(data)
    
    def fit(self, train_data, train_labels, epochs=10, 
            val_data=None, val_labels=None, verbose=True):
        history = {'train_loss': [], 'train_acc': [], 'val_acc': []}
        
        for epoch in range(epochs):
            loss, acc = self.train_epoch(train_data, train_labels)
            history['train_loss'].append(loss)
            history['train_acc'].append(acc)
            
            if val_data is not None:
                val_acc = self.evaluate(val_data, val_labels)
                history['val_acc'].append(val_acc)
            
            if verbose and ((epoch + 1) % 5 == 0 or epoch == 0):
                msg = f"Epoch {epoch+1:3d}: Loss={loss:.4f}, Acc={acc:.3f}"
                if val_data:
                    msg += f", Val={val_acc:.3f}"
                print(msg)
        
        return history
    
    def evaluate(self, data, labels):
        correct = sum(
            1 for inputs, label in zip(data, labels)
            if self.network.predict(inputs) == label
        )
        return correct / len(data)

---

## Part 8: Demonstration

In [ ]:
def generate_data(num_samples, input_size, num_classes):
    data = []
    labels = []
    chunk_size = input_size // num_classes
    
    for _ in range(num_samples):
        inputs = [random.random() for _ in range(input_size)]
        chunk_sums = [
            sum(inputs[c * chunk_size:(c + 1) * chunk_size])
            for c in range(num_classes)
        ]
        label = chunk_sums.index(max(chunk_sums))
        data.append(inputs)
        labels.append(label)
    
    return data, labels

In [ ]:
# Configuration
INPUT_SIZE = 20
HIDDEN_SIZE = 16
NUM_CLASSES = 4

# Build network using component-based design
layers = [
    DenseLayer(INPUT_SIZE, HIDDEN_SIZE, ReLU(), init_method='he'),
    OutputLayer(HIDDEN_SIZE, NUM_CLASSES)
]

network = NeuralNetwork(layers)
network.summary()

In [ ]:
# Generate data
train_data, train_labels = generate_data(500, INPUT_SIZE, NUM_CLASSES)
test_data, test_labels = generate_data(100, INPUT_SIZE, NUM_CLASSES)

# Create optimiser and trainer
optimiser = SGD(layers, learning_rate=0.1)
trainer = Trainer(network, optimiser)

print("Training with SGD...\n")
history = trainer.fit(
    train_data, train_labels,
    epochs=20,
    val_data=test_data,
    val_labels=test_labels
)

In [ ]:
# Compare SGD vs SGD with Momentum
print("\nComparing optimisers:")
print("=" * 50)

for opt_class, opt_name in [(SGD, "SGD"), (SGDMomentum, "SGD+Momentum")]:
    # Fresh network
    layers = [
        DenseLayer(INPUT_SIZE, HIDDEN_SIZE, ReLU()),
        OutputLayer(HIDDEN_SIZE, NUM_CLASSES)
    ]
    net = NeuralNetwork(layers)
    
    if opt_class == SGDMomentum:
        opt = opt_class(layers, learning_rate=0.1, momentum=0.9)
    else:
        opt = opt_class(layers, learning_rate=0.1)
    
    trainer = Trainer(net, opt)
    trainer.fit(train_data, train_labels, epochs=15, verbose=False)
    
    test_acc = trainer.evaluate(test_data, test_labels)
    print(f"{opt_name:15s}: Test accuracy = {test_acc:.3f}")

In [ ]:
# Compare different activations
print("\nComparing activation functions:")
print("=" * 50)

activations = [ReLU(), LeakyReLU(0.01), Sigmoid(), Tanh()]

for activation in activations:
    layers = [
        DenseLayer(INPUT_SIZE, HIDDEN_SIZE, activation),
        OutputLayer(HIDDEN_SIZE, NUM_CLASSES)
    ]
    net = NeuralNetwork(layers)
    opt = SGD(layers, learning_rate=0.1)
    trainer = Trainer(net, opt)
    
    trainer.fit(train_data, train_labels, epochs=15, verbose=False)
    test_acc = trainer.evaluate(test_data, test_labels)
    
    print(f"{activation.name:20s}: Test accuracy = {test_acc:.3f}")

---

## Summary

### Component Relationships

```
┌──────────────────────────────────────────────────────────┐
│                        Trainer                           │
│  ┌─────────────────┐     ┌─────────────────────────────┐│
│  │ NeuralNetwork   │     │ Optimiser (SGD/Momentum)    ││
│  │  ┌───────────┐  │     │                             ││
│  │  │ [layers]  │←─┼─────┤                             ││
│  │  └───────────┘  │     └─────────────────────────────┘│
│  └─────────────────┘                                    │
└──────────────────────────────────────────────────────────┘

DenseLayer:
┌────────────────────────────┐
│ weights, biases            │
│ ┌────────────────────────┐ │
│ │ Activation (Strategy)  │ │
│ │ - ReLU                 │ │
│ │ - Sigmoid              │ │
│ │ - etc.                 │ │
│ └────────────────────────┘ │
└────────────────────────────┘
```

### Design Patterns Used

| Pattern | Where | Benefit |
|---------|-------|--------|
| Strategy | Activation functions | Easy to swap/add activations |
| Strategy | Optimisers | Easy to swap/add optimisers |
| Composition | Network has layers | Flexible network construction |
| Template Method | Layer backward pass | Common structure, variable details |

### For MNIST

```python
layers = [
    DenseLayer(784, 256, ReLU()),
    DenseLayer(256, 128, ReLU()),
    OutputLayer(128, 10)
]
network = NeuralNetwork(layers)
optimiser = SGDMomentum(layers, learning_rate=0.01, momentum=0.9)
trainer = Trainer(network, optimiser)
trainer.fit(train_images, train_labels, epochs=10)
```

---

**Next notebook**: See the full object-oriented design with Node/Connection classes.